In [ ]:
from pyspark.sql.functions import *

In [ ]:
spark.conf.set(
    "fs.azure.account.key.<adls-storage-account>.dfs.core.windows.net",
    "<access-keys>"
)


In [ ]:
parquet_path = "abfss://<container-name>@<adls-storage-account>.dfs.core.windows.net/silver"

In [ ]:
gold_df = spark.read.option("header", True).parquet(parquet_path)

In [ ]:
gold_df.printSchema()

In [ ]:
spark.conf.set("spark.sql.catalog.iceberg_catalog", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.iceberg_catalog.type", "hadoop")
spark.conf.set(
    "spark.sql.catalog.iceberg_catalog.warehouse",
    "abfss://<adls-container-name>@<adls-storage-account-name>.dfs.core.windows.net/iceberg_warehouse/"
)

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS iceberg_catalog.iceberg_db")

In [ ]:
spark.sql("DROP TABLE IF EXISTS iceberg_catalog.iceberg_db.parking_iceberg")

In [ ]:
gold_df.writeTo("iceberg_catalog.iceberg_db.parking_iceberg").createOrReplace()

In [ ]:
%sql

SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg;

In [ ]:
%sql
DESCRIBE EXTENDED iceberg_catalog.iceberg_db.parking_iceberg;


##### Adding Columns

%sql

ALTER TABLE iceberg_catalog.iceberg_db.parking_iceberg
ADD COLUMN is_electric BOOLEAN;


In [ ]:
updated_df = gold_df.withColumn(
    "is_electric",
    when(upper(gold_df.Vehicle_Make) == "TESLA", True).otherwise(False)
)

In [ ]:
updated_df.writeTo("iceberg_catalog.iceberg_db.parking_iceberg").overwritePartitions()


In [ ]:
columns = [field.name for field in spark.table("iceberg_catalog.iceberg_db.parking_iceberg").schema]


In [ ]:
print(columns)


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.history;


In [ ]:
%sql
DESCRIBE EXTENDED iceberg_catalog.iceberg_db.parking_iceberg;


In [ ]:
%sql
-- Query new column
SELECT Summons_Number, Vehicle_Make, is_electric, Registration_State, Violation_County
FROM iceberg_catalog.iceberg_db.parking_iceberg
WHERE is_electric = TRUE;


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots
ORDER BY committed_at DESC
LIMIT 10;


In [ ]:
SHOW TABLES IN iceberg_catalog.iceberg_db;

In [ ]:
%sql
-- Inspect manifest files
SELECT snapshot_id, manifest_list
FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots;


##### Rename Columns

%sql
ALTER TABLE iceberg_catalog.iceberg_db.parking_iceberg
RENAME COLUMN Vehicle_Color TO exterior_color;


In [ ]:
%sql
SELECT Summons_Number, exterior_color
FROM iceberg_catalog.iceberg_db.parking_iceberg
WHERE exterior_color = 'BLUE';


In [ ]:
%sql
-- Verify schema evolution in metadata
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.history;

-- Note "RENAME_COLUMN" operation with old/new names


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots
ORDER BY committed_at DESC
LIMIT 10;

%sql
-- Check manifest entries for column ID consistency
SELECT snapshot_id
FROM iceberg_catalog.iceberg_db.parking_iceberg.history;

##### Drop Columns

%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots
ORDER BY committed_at DESC;

In [ ]:
%sql
-- Drop column
ALTER TABLE iceberg_catalog.iceberg_db.parking_iceberg
DROP COLUMN Unregistered_Vehicle;


In [ ]:
%sql
-- Attempt query
SELECT Unregistered_Vehicle FROM iceberg_catalog.iceberg_db.parking_iceberg;


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.history;


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots
ORDER BY committed_at DESC;


In [ ]:
%sql
-- Validate physical data still exists
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.files;


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots;


In [ ]:
%sql
CREATE OR REPLACE TABLE iceberg_catalog.iceberg_db.restored_iceberg_table AS
SELECT * FROM iceberg_catalog.iceberg_db.parking_iceberg VERSION AS OF <old-snapshot_id>;


In [ ]:
%sql
SELECT * FROM iceberg_catalog.iceberg_db.restored_iceberg_table;


In [ ]:
%sql
-- Audit schema changes
SELECT
  snapshot_id,
  operation,
  summary['schema-changes']
FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots;


#### Changed Data Type

%sql
DESCRIBE iceberg_catalog.iceberg_db.parking_iceberg;


In [ ]:
%sql
SELECT Vehicle_Year, COUNT(*)
FROM iceberg_catalog.iceberg_db.parking_iceberg
GROUP BY Vehicle_Year
ORDER BY COUNT(*) DESC
LIMIT 5;


In [ ]:
%sql
ALTER TABLE iceberg_catalog.iceberg_db.parking_iceberg
ALTER COLUMN Vehicle_Year TYPE long;


In [ ]:
%sql
-- Verify new type and query
DESCRIBE iceberg_catalog.iceberg_db.parking_iceberg Vehicle_Year;


In [ ]:
%sql
SELECT
  operation,
  summary['rewrote-data-files'] AS rewritten_files
FROM iceberg_catalog.iceberg_db.parking_iceberg.snapshots
ORDER BY committed_at DESC
LIMIT 1;


In [ ]:
%sql
-- Inspect metadata evolution
SELECT *
FROM iceberg_catalog.iceberg_db.parking_iceberg.metadata_log_entries;
